# Parameters

In [ ]:
dollars_per_ton_co2 = 300 # metric tons
location = "Minneapolis"
square_footage = 1200
current_heating_system = "Natural Gas"
chosen_set_point = 60
price_per_kWh_electric = 0.12
price_per_therm_secondary = None
rebate_type = 'partial' # full, partial, or none
winner_output_path = 'winner.json'
unified_r_factor = 3.5 # this should range from 2.5 to 5

In [189]:
price_per_kWh_electric = 0.12 if price_per_kWh_electric is None else price_per_kWh_electric
price_per_therm_secondary = 1.20 if price_per_therm_secondary is None else price_per_therm_secondary

# Imports

In [174]:
! pip install papermill

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [papermill]11 [nbclient]a]


In [64]:
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d

import pyarrow
import matplotlib.pyplot as plt
import re
import pandas as pd
import os

In [65]:
data=pd.read_csv("data/raw/buildstock/MN_upgrade0.csv", dtype={'MN_upgrade0.csv': 'str'})

/tmp/ipykernel_7755/105414082.py:1: DtypeWarning: Columns (5,144,151,152,359,360) have mixed types. Specify dtype option on import or set low_memory=False.
  data=pd.read_csv("data/raw/buildstock/MN_upgrade0.csv", dtype={'MN_upgrade0.csv': 'str'})


# Constants
Daddy's made you some constants.

In [66]:
UNFINISHED_R_VALUES = {
    "floor_r": 2.0,  # 4" concrete slab + ground contact + air film
    "foundation_wall_r": 2.0,  # 8" poured concrete wall + air films
    "ceiling_r": 2.5,  # 5/8" drywall/joists + still air film
    "rim_joist_r": 2.5,  # 1.5" wood rim board + sheathing + air films
    "roof_r": 3.0,  # 5/8" OSB decking + asphalt shingles + air films
}

columns_of_interest = [
    "MN_upgrade0.csv",
    "in.city",
    "in.sqft..ft2",
    'in.heating_fuel',
    'in.heating_setpoint',
    "in.insulation_ceiling",
    "in.insulation_floor",
    "in.insulation_foundation_wall",
    "in.insulation_rim_joist",
    "in.insulation_roof",
    "in.geometry_building_type_recs",
    "out.params.size_heat_pump_backup_primary..kbtu_per_hr",
    "out.params.size_heating_system_primary..kbtu_per_hr",
    "out.params.size_heating_system_secondary..kbtu_per_hr"
]

column_renames = {
    "MN_upgrade0.csv": "building_id",
    "in.city": "city",
    "in.sqft..ft2": 'area',
    "in.heating_fuel": 'heating_fuel',
    "in.heating_setpoint": 'heating_setpoint',
    "in.insulation_ceiling": 'ceiling_r',
    "in.insulation_floor": 'floor_r',
    "in.insulation_foundation_wall": 'foundation_wall_r',
    "in.insulation_rim_joist": 'rim_joist_r',
    "in.insulation_roof": 'roof_r',
    "in.geometry_building_type_recs": 'building_type',
    "out.params.size_heat_pump_backup_primary..kbtu_per_hr": 'heat_pump_backup_primary_size',
    "out.params.size_heating_system_primary..kbtu_per_hr": 'heating_system_primary_size',
    "out.params.size_heating_system_secondary..kbtu_per_hr": 'heating_system_secondary_size'
}

df = data[columns_of_interest].copy()
df.rename(columns=column_renames, inplace=True)
df['building_id'] = df['building_id'].astype('str')

# filter file down to only the columns of interest and only the rows for Minneapolis, MN
df = df.where(
    (df["city"] == "MN, Minneapolis")
    & (df["building_type"] == "Single-Family Detached" ), axis=0
).dropna()

In [67]:
df['building_id'].unique()

array(['1641', '6250', '9572', '13087', '13786', '22043', '27407',
       '30507', '31874', '32331', '35520', '38425', '40952', '45931',
       '47534', '57625', '65224', '67464', '72940', '81997', '82186',
       '82862', '83178', '86765', '90349', '92004', '94083', '98308',
       '98992', '100745', '101421', '103386', '103928', '105194',
       '108104', '111805', '125109', '125389', '133093', '138909',
       '153329', '153974', '157454', '159647', '161029', '161714',
       '165633', '167877', '171825', '173238', '179657', '181617',
       '181832', '182925', '190720', '191361', '198065', '200457',
       '205050', '205881', '209362', '213254', '213931', '223019',
       '227783', '231096', '234796', '235665', '235990', '236632',
       '237987', '240687', '241727', '245316', '252437', '253128',
       '253632', '253964', '254216', '254612', '257267', '261019',
       '268305', '269531', '272747', '272790', '274938', '276276',
       '281120', '283116', '284960', '286536', '292265

# Functions

In [162]:
# Standard baseline R-values for uninsulated (unfinished) building elements

def c2f(celsius: float) -> float:
    """Convert Celsius to Fahrenheit."""
    return celsius * 9.0 / 5.0 + 32.0

def get_parquet_file_path(state_abbreviation: str, building_id: str) -> str:
    """Returns the directory path for parquet files based on the state abbreviation."""
    return f"data/raw/nrel_parquet/{state_abbreviation.lower()}/{building_id}-0.parquet"

def parse_r_value(val: str, default_r: float, col_name: str = "not specified") -> float:
    """Extracts the numeric float directly following 'r-' or 'R-'.
    Defaults to default_r if 'r-<number>' isn't present or val is missing.
    """
    if pd.isna(val):
        return default_r

    # Looks specifically for 'r-' or 'R-' followed by digits/decimals
    match = re.search(r'r-(\d+(?:\.\d+)?)', str(val), flags=re.IGNORECASE)

    return float(match.group(1)) if match else default_r


def load_neep_machine_profiles(
    csv_path: str = "data/raw/heat_pumps/neep_ccashp_heating_performance.csv",
    cop_col: str = 'cop_rated'
) -> dict[str, dict]:
    """Reads NEEP AHRI CSV data dynamically into temperature rating dictionaries.

    Args:
        csv_path: Path to NEEP CSV file.
        cop_col: Column to use for efficiency ('cop_max', 'cop_rated', or
          'cop_min'). Default 'cop_max' provides continuous coverage down to
          -22°F.
    """
    df = pd.read_csv(csv_path, skiprows=1)
    profiles = {}

    for (brand, outdoor_unit, pid), group in df.groupby(
        ['brand', 'outdoor_unit', 'product_id']
    ):
        group = group.sort_values('outdoor_F')

        cop_series = group[cop_col].fillna(group['cop_min'])

        # Extract temperature-mapped dictionaries directly from CSV columns
        cap_ratings = dict(
            zip(
                group['outdoor_F'].astype(float),
                group['capacity_max_btuh'].astype(float),
            )
        )
        cop_ratings = dict(
            zip(group['outdoor_F'].astype(float), cop_series.astype(float))
        )

        profile_label = f'AHRI {pid}'

        profiles[profile_label] = {
            'cop_ratings': cop_ratings,
            'capacity_ratings': cap_ratings,
            'nominal_tons': cap_ratings.get(47.0, 0.0) / 12000.0,
            'brand': brand,
            'outdoor_unit': outdoor_unit,
            'product_id': pid,
        }

    return profiles


def calculate_unified_r(
    df: pd.DataFrame, area_weights: dict = None
) -> pd.DataFrame:
    """Parses string R-values to floats and computes a unified parallel envelope R-value."""
    df_out = df.copy()
    r_cols = list(UNFINISHED_R_VALUES.keys())
    num_cols = []

    # Parse columns into numeric floats
    for col in r_cols:
        num_col = f"{col}_float"
        default = UNFINISHED_R_VALUES[col]
        df_out[num_col] = df_out[col].apply(
            lambda x: parse_r_value(x, default, col_name=col)
        )
        num_cols.append(num_col)

    # Parallel Heat Flow Calculation: U_total = sum(U_i) -> R_unified = 1 / U_avg
    u_values = 1.0 / df_out[num_cols].values

    if area_weights:
        # Area-weighted harmonic mean: R_unified = sum(A_i) / sum(A_i / R_i)
        weights = np.array([area_weights[col] for col in r_cols])
        df_out["unified_r_value"] = weights.sum() / (u_values @ weights)
    else:
        # Unweighted harmonic mean across the 5 assemblies
        df_out["unified_r_value"] = len(r_cols) / u_values.sum(axis=1)

    return df_out


def capital_cost(tons):
    """Installed cost ($) of a heat pump. size_kbtuh = nominal capacity in kBtu/h (valid 18-60)."""
    # size_kbtuh = tons
    if tons <= 0:
        return 0
    return 1075.2 * tons + 1193


def get_rebate(capital_cost: float, type: float):
    if type == 'full':
        # for income < 80% of AMI
        return min(capital_cost, 8000)
    elif type == 'partial':
        # for income between 80% and 150% of AMI
        return min(capital_cost/2, 4000)
    else:
        return -0

def get_payback_period(annual_savings: float, capital_cost: float) -> float:
    return capital_cost / annual_savings if annual_savings > 0 else float('inf')


df['ceiling_r'].unique()


df = calculate_unified_r(df)

In [69]:
df['building_type'].unique()

array(['Single-Family Detached'], dtype=object)

In [70]:
def find_closest_row(df, heating_fuel, heating_setpoint, unified_r, area):
    numeric_columns = ["heating_setpoint", "unified_r_value", "area"]
    numeric_values = df[numeric_columns].copy()
    numeric_values["heating_setpoint"] = pd.to_numeric(
        numeric_values["heating_setpoint"]
        .astype("string")
        .str.extract(r"([-+]?\d+(?:\.\d+)?)", expand=False),
        errors="coerce",
    )
    numeric_values[["unified_r_value", "area"]] = numeric_values[
        ["unified_r_value", "area"]
    ].apply(pd.to_numeric, errors="coerce")

    candidates = df[df["heating_fuel"] == heating_fuel]
    if candidates.empty:
        raise ValueError(f"No rows found for heating_fuel={heating_fuel!r}")

    candidate_values = numeric_values.loc[candidates.index].dropna()
    if candidate_values.empty:
        raise ValueError(f"No numeric rows found for heating_fuel={heating_fuel!r}")

    setpoint = pd.to_numeric(
        pd.Series([heating_setpoint], dtype="string").str.extract(
            r"([-+]?\d+(?:\.\d+)?)", expand=False
        ).iloc[0],
        errors="coerce",
    )
    target = pd.Series([setpoint, unified_r, area], index=numeric_columns, dtype=float)
    scales = numeric_values.std().replace(0, 1)
    distances = (((candidate_values - target) / scales) ** 2).sum(axis=1)
    return df.loc[distances.idxmin()]


In [ ]:
closest_building = find_closest_row(
    df,
    'Natural Gas',
    chosen_set_point,
    unified_r_factor,
    square_footage
)
closest_building

building_id                                       171825
city                                     MN, Minneapolis
area                                              1698.0
heating_fuel                                 Natural Gas
heating_setpoint                                     68F
ceiling_r                                           R-30
floor_r                                      Uninsulated
foundation_wall_r                    Wall R-10, Exterior
rim_joist_r                               R-10, Exterior
roof_r                           Unfinished, Uninsulated
building_type                     Single-Family Detached
heat_pump_backup_primary_size                        0.0
heating_system_primary_size                        34.07
heating_system_secondary_size                        0.0
floor_r_float                                        2.0
foundation_wall_r_float                             10.0
ceiling_r_float                                     30.0
rim_joist_r_float              

In [72]:
# os.path.isfile(get_parquet_file_path("MN", '274938'))


In [ ]:
df_ts = pd.read_parquet(
    get_parquet_file_path(
        "MN",
        closest_building['building_id']
)
df_ts.set_index('timestamp', inplace=True)

cols = list(df_ts.columns)
cols.sort()
cols

# The column we care about is:
# 'out.load.heating.energy_delivered..kbtu'

'out.load.heating.energy_delivered..kbtu'

In [86]:
df_temp = pd.read_csv("./data/raw/weather/mn.csv", index_col="date_time")

In [77]:
df_temp['Dry Bulb Temperature [°C]']

0      -23.99
1      -23.99
2      -24.04
3      -24.33
4      -25.00
        ...  
8755    -7.28
8756    -8.43
8757    -9.64
8758   -11.47
8759   -11.47
Name: Dry Bulb Temperature [°C], Length: 8760, dtype: float64

# Model

In [156]:
def run_thermal_hp_sim(
    df_hourly: pd.DataFrame,
    setpoint_f: float = 68.0,
    cop_ratings: dict[float, float] | None = None,
    capacity_ratings: dict[float, float] | None = None,
    hp_capacity_tons: float = 3.0,
    gas_furnace_eff: float = 0.80,
    elec_rate_per_kwh: float = 0.15,
    gas_rate_per_therm: float = 1.20,
    grid_co2_kg_per_kwh: float = 0.45,
    gas_co2_kg_per_therm: float = 5.30,
) -> tuple[pd.DataFrame, dict]:
    """Runs an hourly simulation matching heat pump & gas backup directly to external heating demand."""
    # 1. Provide default NEEP/AHRI benchmark rating curves if none supplied
    # Check if a valid, non-zero heat pump capacity was provided
    if (
        capacity_ratings
        and cop_ratings
        and max(capacity_ratings.values()) > 0.0
    ):
        temps_cop = np.array(sorted(cop_ratings.keys()))
        vals_cop = np.array([cop_ratings[t] for t in temps_cop])

        temps_cap = np.array(sorted(capacity_ratings.keys()))
        vals_cap = np.array([capacity_ratings[t] for t in temps_cap])

        get_cop_interp = interp1d(
            temps_cop,
            vals_cop,
            kind="linear",
            bounds_error=False,
            fill_value=(vals_cop[0], vals_cop[-1]),
        )
        get_cap_interp = interp1d(
            temps_cap,
            vals_cap,
            kind="linear",
            bounds_error=False,
            fill_value=(vals_cap[0], vals_cap[-1]),
        )
        has_hp = True
    else:
        has_hp = False

    # 3. Setup tracking arrays
    n_hours = len(df_hourly)
    q_hp_delivered = np.zeros(n_hours)
    q_aux_delivered = np.zeros(n_hours)
    kwh_hp = np.zeros(n_hours)
    therms_aux = np.zeros(n_hours)
    hvac_action = ['OFF'] * n_hours

    # 4. Hourly Simulation Loop over ALL hours
    for t in range(n_hours):
        t_amb = df_hourly['t_amb_f'].iloc[t]
        heat_needed = df_hourly['heating_demand'].iloc[t]

        if heat_needed > 0 and not np.isnan(t_amb):
            if has_hp:
                # Look up interpolated COP and max capacity for current outdoor temp
                cop_current = float(get_cop_interp(t_amb))
                avail_hp_cap = float(get_cap_interp(t_amb))

                # Heat delivered split
                hp_heat = min(heat_needed, avail_hp_cap)
                aux_heat = max(0.0, heat_needed - hp_heat)
                # Energy inputs
                hp_elec_kwh = (hp_heat / max(1.0, cop_current)) / 3412.14
            else:
                hp_heat = 0.0
                aux_heat = heat_needed
                hp_elec_kwh = 0.0

            gas_therms = (aux_heat / gas_furnace_eff) / 100000.0

            if hp_heat > 0 and aux_heat > 0:
                hvac_action[t] = 'HP_AND_GAS_AUX'
            elif hp_heat > 0:
                hvac_action[t] = 'HP_ONLY'
            elif aux_heat > 0:
                hvac_action[t] = 'GAS_AUX_ONLY'
        else:
            hp_heat = 0.0
            aux_heat = 0.0
            hp_elec_kwh = 0.0
            gas_therms = 0.0
            hvac_action[t] = 'OFF'

        q_hp_delivered[t] = hp_heat
        q_aux_delivered[t] = aux_heat
        kwh_hp[t] = hp_elec_kwh
        therms_aux[t] = gas_therms

    # 5. Format Output Dataframe
    df_res = df_hourly.copy()
    df_res['t_in_f'] = setpoint_f  # Demand is satisfied to maintain setpoint
    df_res['hvac_action'] = hvac_action
    df_res['q_hp_btu'] = q_hp_delivered
    df_res['q_aux_btu'] = q_aux_delivered
    df_res['kwh_hp'] = kwh_hp
    df_res['therms_aux'] = therms_aux

    df_res['elec_cost_usd'] = df_res['kwh_hp'] * elec_rate_per_kwh
    df_res['gas_cost_usd'] = df_res['therms_aux'] * gas_rate_per_therm
    df_res['total_operating_cost'] = (
        df_res['elec_cost_usd'] + df_res['gas_cost_usd']
    )

    df_res['co2_elec_kg'] = df_res['kwh_hp'] * grid_co2_kg_per_kwh
    df_res['co2_gas_kg'] = df_res['therms_aux'] * gas_co2_kg_per_therm
    df_res['total_co2_kg'] = df_res['co2_elec_kg'] + df_res['co2_gas_kg']

    summary = {
        'total_elec_kwh': df_res['kwh_hp'].sum(),
        'total_gas_therms': df_res['therms_aux'].sum(),
        'total_operating_cost': df_res['total_operating_cost'].sum(),
        'total_co2_kg': df_res['total_co2_kg'].sum(),
        'hp_coverage_pct': (
            df_res['q_hp_btu'].sum()
            / max(1.0, (df_res['q_hp_btu'].sum() + df_res['q_aux_btu'].sum()))
        )
        * 100,
        'min_indoor_temp_f': df_res['t_in_f'].min(),
    }

    return df_res, summary

# 1. Convert outdoor temperature to °F and extract raw values
t_amb_vals = df_temp['Dry Bulb Temperature [°C]'].apply(c2f).values

# 2. Resample 15-minute demand to hourly sums (kBTU -> BTU) and extract raw values
heating_demand_vals = (
    df_ts['out.load.heating.energy_delivered..kbtu']
    .resample('h')
    .sum()
    * 1000.0
).values

# 3. Handle length mismatch: trim both to exact 8760 hours
min_len = min(len(t_amb_vals), len(heating_demand_vals), 8760)
t_amb_vals = t_amb_vals[:min_len]
heating_demand_vals = heating_demand_vals[:min_len]

# 4. Create clean hourly DatetimeIndex
clean_hours = pd.date_range('2018-01-01 00:00', periods=min_len, freq='h')

# 5. Build input DataFrame without index mismatch
df_input = pd.DataFrame(
    {'t_amb_f': t_amb_vals, 'heating_demand': heating_demand_vals},
    index=clean_hours,
)

# Quick check: print shape to confirm non-empty
print(f'df_input created successfully with shape: {df_input.shape}')

# 6. Run simulation
user_cops = {47.0: 3.80, 17.0: 2.60, 5.0: 2.00, -22.0: 1.15}
user_caps = {
    47.0: 36000.0,
    17.0: 32000.0,
    5.0: 27000.0,
    -22.0: 14000.0,
}

results_df, summary = run_thermal_hp_sim(
    df_hourly=df_input,
    cop_ratings=user_cops,
    capacity_ratings=user_caps,
    elec_rate_per_kwh=0.17,
    gas_rate_per_therm=1.20,
)

print('\n=== Annual Summary ===')
for k, v in summary.items():
    print(f'{k}: {v:,.2f}')

print('\n=== Sample Hours Sanity Check ===')
print(
    results_df[
        [
            't_amb_f',
            't_in_f',
            'hvac_action',
            'q_hp_btu',
            'q_aux_btu',
            'kwh_hp',
            'therms_aux',
        ]
    ].head(10)
)

df_input

df_input created successfully with shape: (8760, 2)

=== Annual Summary ===
total_elec_kwh: 11,490.74
total_gas_therms: 312.01
total_operating_cost: 2,327.84
total_co2_kg: 6,824.49
hp_coverage_pct: 81.19
min_indoor_temp_f: 68.00

=== Sample Hours Sanity Check ===
                     t_amb_f  t_in_f     hvac_action      q_hp_btu  \
2018-01-01 00:00:00  -11.182    68.0  HP_AND_GAS_AUX  19208.666667   
2018-01-01 01:00:00  -11.182    68.0  HP_AND_GAS_AUX  19208.666667   
2018-01-01 02:00:00  -11.272    68.0  HP_AND_GAS_AUX  19165.333333   
2018-01-01 03:00:00  -11.794    68.0  HP_AND_GAS_AUX  18914.000000   
2018-01-01 04:00:00  -13.000    68.0  HP_AND_GAS_AUX  18333.333333   
2018-01-01 05:00:00  -13.000    68.0  HP_AND_GAS_AUX  18333.333333   
2018-01-01 06:00:00  -13.756    68.0  HP_AND_GAS_AUX  17969.333333   
2018-01-01 07:00:00  -14.728    68.0  HP_AND_GAS_AUX  17501.333333   
2018-01-01 08:00:00  -14.260    68.0  HP_AND_GAS_AUX  17726.666667   
2018-01-01 09:00:00  -11.236    68.0

,t_amb_f,heating_demand
2018-01-01 00:00:00,-11.182,24964.900391
2018-01-01 01:00:00,-11.182,50496.011719
2018-01-01 02:00:00,-11.272,72522.523438
2018-01-01 03:00:00,-11.794,73223.835938
2018-01-01 04:00:00,-13.000,73866.851562
...,...,...
2018-12-31 19:00:00,18.896,39319.730469
2018-12-31 20:00:00,16.826,39977.730469
2018-12-31 21:00:00,14.648,42446.000000
2018-12-31 22:00:00,11.354,41509.320312


# Parameter search

In [ ]:
# Parse all 21 machine profiles dynamically from CSV
profiles = load_neep_machine_profiles(
    cop_col='cop_max'
)
print(f'Loaded {len(profiles)} machine profiles from CSV.\n')

# Construct input heating demand & ambient temp DataFrame
t_amb_vals = df_temp['Dry Bulb Temperature [°C]'].apply(c2f).values
heating_demand_vals = (
    df_ts['out.load.heating.energy_delivered..kbtu']
    .resample('H')
    .sum()
    * 1000.0
).values

min_len = min(len(t_amb_vals), len(heating_demand_vals), 8760)
clean_hours = pd.date_range('2018-01-01 00:00', periods=min_len, freq='h')

df_input = pd.DataFrame(
    {
        't_amb_f': t_amb_vals[:min_len],
        'heating_demand': heating_demand_vals[:min_len],
    },
    index=clean_hours,
)

results_list = []

# 1. Run Baseline: 100% Gas Furnace (no COP/Capacity ratings passed)
_, base_summary = run_thermal_hp_sim(
    df_hourly=df_input,
    cop_ratings=None,
    capacity_ratings=None,
    elec_rate_per_kwh=price_per_kWh_electric,
    gas_rate_per_therm=price_per_therm_secondary
)
base_summary['machine_profile'] = 'NG only'
base_summary['nominal_tons'] = 0.0
results_list.append(base_summary)

# 2. Run all CSV Machine Profiles
for profile_label, pdata in profiles.items():
    _, summary = run_thermal_hp_sim(
        df_hourly=df_input,
        cop_ratings=pdata['cop_ratings'],
        capacity_ratings=pdata['capacity_ratings'],
        elec_rate_per_kwh=price_per_kWh_electric,
        gas_rate_per_therm=price_per_therm_secondary
    )

    summary['machine_profile'] = profile_label
    summary['nominal_tons'] = pdata['nominal_tons']
    results_list.append(summary)

# Display compiled results matrix
df_results = pd.DataFrame(results_list)
cols = [
    'machine_profile',
    'nominal_tons',
    'hp_coverage_pct',
    'total_elec_kwh',
    'total_gas_therms',
    'total_operating_cost',
    'total_co2_kg',
]

df_results = df_results[cols].sort_values(by='nominal_tons')

# apply capital_cost() to add column to df_results:
df_results['capital_cost_usd'] = df_results['nominal_tons'].apply(capital_cost)

# apply get_rebate() to add column to df_results:
df_results['rebate_usd'] = df_results['nominal_tons'].apply(
    lambda tons: get_rebate(capital_cost(tons), rebate_type)
)

df_results['grand_operating_cost'] = df_results['total_operating_cost'] + (df_results['total_co2_kg']*dollars_per_ton_co2)/1000

print('=== DYNAMIC NEEP MACHINE PERFORMANCE MATRIX ===')
print(df_results.to_string(index=False))

Loaded 21 machine profiles from CSV.



/tmp/ipykernel_7755/2500494220.py:11: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .resample('H')


=== DYNAMIC NEEP MACHINE PERFORMANCE MATRIX ===
machine_profile  nominal_tons  hp_coverage_pct  total_elec_kwh  total_gas_therms  total_operating_cost  total_co2_kg  capital_cost_usd  rebate_usd  grand_operating_cost
        NG only      0.000000         0.000000        0.000000       1658.486406           1990.183687   8789.977953              0.00        0.00           4627.177073
    AHRI 403896      1.750000        63.854749     9982.548077        599.464072           1917.262655   7669.306214           3074.60     1537.30           4218.054519
    AHRI 391920      1.833333        65.786284    10061.077855        567.429833           1888.245142   7534.863149           3164.20     1582.10           4148.704087
    AHRI 408064      2.333333        75.909507    11614.383718        399.537555           1873.171112   7344.021714           3701.80     1850.90           4076.377626
    AHRI 438606      2.333333        76.266040    11637.092065        393.624499           1868.800446   73

# Heat pump selection

In [ ]:
# 1. Compute annual grand operating savings relative to Natural Gas baseline
ng_baseline_cost = df_results.loc[
    df_results['nominal_tons'] == 0.0, 'grand_operating_cost'
].values[0]

df_results['grand_operating_savings'] = (
    ng_baseline_cost - df_results['grand_operating_cost']
)

# 2. Compute net out-of-pocket capital cost (Capital Cost minus Rebate)
df_results['net_capital_cost_usd'] = (
    df_results['capital_cost_usd'] - df_results['rebate_usd']
)

# 3. Filter heat pumps with positive annual savings (using .copy() to prevent SettingWithCopyWarning)
df_res_w_sav = df_results.loc[
    (df_results['nominal_tons'] > 0.0)
    & (df_results['grand_operating_savings'] > 0),
    :,
].copy()

# 4. Calculate simple payback period (years)
df_res_w_sav['grand_payback'] = (
    df_res_w_sav['net_capital_cost_usd']
    / df_res_w_sav['grand_operating_savings']
)

# 5. Select winner with the lowest Grand Operating Cost
df_winner = df_res_w_sav.loc[df_res_w_sav['grand_operating_cost'].idxmin()]

# write results to CSV for further analysis
df_results.to_csv('heat_pump_selection_results.csv', index=False)

# write winner to json and format it nicely:
df_winner.to_json(winner_output_path, orient='records', indent=4)

# indicate the selected building:


# Optional: Print comparison summary against Natural Gas
print('=== OPTIMAL HEAT PUMP SELECTION ===')
print(f"Winning Machine:           {df_winner['machine_profile']}")
print(f"Capacity Tonnage:          {df_winner['nominal_tons']:.2f} Tons")
print(f"HP Heating Coverage:       {df_winner['hp_coverage_pct']:.1f}%")
print(f"NG Baseline Grand Cost:    ${ng_baseline_cost:,.2f}/yr")
print(
    f"Winning HP Grand Cost:     ${df_winner['grand_operating_cost']:,.2f}/yr"
)
print(
    f"Annual Grand Savings:      ${df_winner['grand_operating_savings']:,.2f}/yr"
)
print(
    f"Net Out-of-Pocket Cost:    ${df_winner['net_capital_cost_usd']:,.2f}"
)

=== OPTIMAL HEAT PUMP SELECTION ===
Winning Machine:           AHRI 438604
Capacity Tonnage:          2.83 Tons
HP Heating Coverage:       84.4%
NG Baseline Grand Cost:    $4,627.18/yr
Winning HP Grand Cost:     $3,923.24/yr
Annual Grand Savings:      $703.93/yr
Net Out-of-Pocket Cost:    $2,119.70
